In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os

def prepare():
    module_path = os.path.abspath(os.path.join('..'))
    if module_path not in sys.path:
        sys.path.append(module_path)

In [ ]:
import torch
import numpy as np
prepare()
from exp_labelcert_binaryclass import run

In [ ]:
model_params = dict(
    label = "GCN", 
    model = "GCN", 
    normalization = "row_normalization",
    activation = "relu",
    depth = 1,
    regularizer = 0.01,
    pred_method = "svm",
    bias = False,
    alpha_tol = 1e-4,
    solver = "qplayer",
)

certificate_params = dict(
    delta = 0.01,
    TimeLimit = 86400,
    LogToConsole = 1,
    OutputFlag = 1,
    Threads = 2,
    Presolve = 2
)

verbosity_params = dict(
    debug_lvl = "warning"
)  

other_params = dict(
    device = "0",
    dtype = torch.float64,
    allow_tf32 = False,
    path_gurobi_license = "path/to/your/gurobi/license"
)

In [ ]:
data_params = dict(
    dataset = "cba",
    learning_setting = "transductive", 
    specification = dict(
        classes = 2,
        n_trn_labeled = 10,
        n_trn_unlabeled = 0,
        n_val = 10,
        n_test = 180,
        sigma = 1,
        avg_within_class_degree = 1.58 * 2,
        avg_between_class_degree = 0.37 * 2,
        K = 1.5,
        m = 2,
        seed = 0 # used to generate the dataset & data split
    )
)

In [ ]:
import pandas as pd
import time

seeds = [0, 1, 2, 3, 4]
delta = 0.3
certificate_params["delta"] = delta

metrics = [
    "accuracy_test",
    "accuracy_trn",
    "accuracy_cert_pois_robust",
    "accuracy_cert_pois_unrobust",
    "delta",
]

summary = []

for seed in seeds:
    data_params["specification"]["seed"] = seed
    
    start_time = time.time()
    result = run(data_params, model_params, certificate_params, verbosity_params, other_params, seed)
    end_time = time.time()
    runtime = round(end_time - start_time, 2)
    
    summary.append({k: result[k] for k in metrics} | {"runtime": runtime})
    
    
df = pd.DataFrame(summary, index=[f"Seed {s}" for s in seeds])
df.index.name = "seed"
df.to_csv(f'results/samplewise/cba-{delta:.2f}.csv', index=True)
df